# R5 · Cited explanations + offline evidence index (Rathi)

**Run this only after the R4 notebook has finished and pushed** (it needs the fine-tuned NLI model on Hugging Face Hub).

**Runtime → Change runtime type → A100 GPU**, switch on **Notebook access** for `GITHUB_TOKEN`, `HF_TOKEN`, `WANDB_KEY` in the 🔑 panel of *this* notebook, then **Runtime → Run all** (about 40 min).

Steps: 1 setup → 2 tests + 10-row checks → 3 notes (200 EN + 50 AR) → 4 faithfulness → 5 build index + recall@5 → 6 upload index → 7 push.

In [ ]:
# 1. Setup
from google.colab import userdata
import os, json
REPO = '/content/twincheck'
if not os.path.exists(REPO):
    !git clone https://github.com/himaallu/Group-Project.git {REPO}
%cd {REPO}
!git config user.name "Rathi"
!git config user.email "335395537+rathiabeth-sudo@users.noreply.github.com"
!git fetch -q origin && git checkout -q rathi/R5-explain-index && git pull -q origin rathi/R5-explain-index
!git merge -q --no-edit origin/rathi/R4-nli   # brings in the R4 outputs

GH = userdata.get('GITHUB_TOKEN').strip()
!git remote set-url --push origin https://{GH}@github.com/himaallu/Group-Project.git
ok = !git ls-remote https://{GH}@github.com/himaallu/Group-Project.git HEAD
print('GitHub token OK' if ok and 'fatal' not in ' '.join(ok) else 'GITHUB_TOKEN rejected: make a new one with the "repo" box ticked')

!pip install -q -U wandb bitsandbytes rank_bm25 faiss-cpu
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
USER = api.whoami()['name']
RERANKER, INDEX_REPO = f'{USER}/twincheck-reranker-v1', f'{USER}/twincheck-index-v1'
# zero-shot NLI for now (R4 v1 scored below it); switch to f'{USER}/twincheck-nli-v2' if v2 wins
os.environ['TWINCHECK_NLI'] = 'MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7'
print('NLI model:', os.environ['TWINCHECK_NLI'], '| reranker:', RERANKER)

## 2. Tests + 10-row checks (5 EN + 5 AR notes, a 10-record index)

In [ ]:
!python -m pytest tests/explain -q
!python -m src.explain.generate --n-en 5 --n-ar 5 --reranker {RERANKER} --out results/R5/test10/notes.jsonl
!python -m src.explain.faithfulness --notes results/R5/test10/notes.jsonl --out-dir results/R5/test10
for line in open('results/R5/test10/notes_scored.jsonl'):
    r = json.loads(line); print(r['language'], r['label'], '|', r['note'], '|', r['sentence_entailed'])
!python -m src.retrieval.build_index build --limit 10 --out-dir models/index_test10
!python -m src.retrieval.build_index eval --out-dir models/index_test10 --n-en 5 --n-ar 5     --reranker {RERANKER} --report results/R5/test10/retrieval_eval.json

## 3. Notes for 200 English + 50 Arabic test claims (Qwen3-8B, ~10 min)

In [ ]:
!python -m src.explain.generate --n-en 200 --n-ar 50 --reranker {RERANKER}

## 4. Faithfulness + citation accuracy (fine-tuned NLI model)

In [ ]:
!python -m src.explain.faithfulness

## 5. Build the offline index (all cleaned snippets, BGE-M3) and measure recall@5

In [ ]:
!python -m src.retrieval.build_index build
!python -m src.retrieval.build_index eval --reranker {RERANKER}

## 6. Upload the index to Hugging Face Hub (dataset repo; ~330 MB, too big for git)

In [ ]:
api.create_repo(INDEX_REPO, repo_type='dataset', exist_ok=True, private=False)
api.upload_folder(folder_path='models/index_v1', repo_id=INDEX_REPO, repo_type='dataset',
                  commit_message='R5: offline evidence library (MultiFC, AVeriTeC, X-FACT AR snippets + BGE-M3 embeddings)')
meta = json.load(open('models/index_v1/meta.json'))
lines = [
    '# index_v1 (R5)',
    '',
    f'Offline evidence library on Hugging Face Hub: https://huggingface.co/datasets/{INDEX_REPO} '
    '(embeddings too big for git).',
    '',
    f'Get it: `python -m src.retrieval.build_index download --repo {INDEX_REPO}`; '
    'then `from src.retrieval.index import search`.',
    '',
    f"{meta['n_snippets']} unique snippets from {meta['n_records']} claims, encoder `{meta['encoder']}`.",
]
open('models/index_v1/README.md', 'w').write('\n'.join(lines) + '\n')
print(open('models/index_v1/README.md').read())

## 7. Push to `rathi/R5-explain-index`

In [ ]:
!git add results/R5/notes.jsonl results/R5/notes_scored.jsonl results/R5/notes_good.jsonl results/R5/notes_bad.jsonl     results/R5/explain_eval.json results/R5/retrieval_eval.json models/index_v1/README.md models/index_v1/meta.json
!git commit -q -m "[R5] Add cited notes, faithfulness eval, offline index meta and recall@5 from Colab run"
!git push -q origin rathi/R5-explain-index && echo PUSHED

**Done.** Tell Claude it finished. Then **Runtime → Disconnect and delete runtime**.